#  Exercise: Density-Based Outlier Detection
In this notebook, you will implement range queries and custom outlier classification to identify noise points in a spatial dataset.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# Load spatial dataset
data_path = Path('../dataset/spatial_density_data.csv')
if not data_path.exists():
    data_path = Path('dataset/spatial_density_data.csv')
if not data_path.exists():
    data_path = Path('Day9/dataset/spatial_density_data.csv')

df = pd.read_csv(data_path)
X = df[['x_coordinate', 'y_coordinate', 'density_metric', 'ambient_noise']].values
X_scaled = StandardScaler().fit_transform(X)
print(f'Loaded dataset shape: {X_scaled.shape}')
print(df.head())


Loaded dataset shape: (200, 4)
   sample_id  x_coordinate  y_coordinate  density_metric  ambient_noise
0          1       15.9934       23.8021         83.9552        12.9385
1          2       14.7235       24.5359         75.4531        10.7143
2          3       16.2954       22.2342         87.0140        10.3937
3          4       18.0461       22.0095         72.9907        12.7237
4          5       14.5317       27.0313         82.9343        11.6648


## Task 1: Identify Noise Points via Local Density
Implement `detect_density_outliers(X, eps, min_samples)` to return a boolean mask where `True` indicates a noise point.

In [2]:
from scipy.spatial.distance import cdist

def detect_density_outliers(X: np.ndarray, eps: float, min_samples: int) -> np.ndarray:
    """
    Computes Euclidean distance matrix and identifies points with fewer
    than min_samples within eps radius.
    Returns boolean array: True for noise points, False for core/border points.
    """
    # 1. Calculate pairwise Euclidean distance matrix
    dist_matrix = cdist(X, X, metric='euclidean')
    
    # 2. Count neighbors within radius eps for each point (including self)
    neighbor_counts = np.sum(dist_matrix <= eps, axis=1)
    
    # 3. Core points have at least min_samples within eps
    core_mask = neighbor_counts >= min_samples
    
    # 4. Outliers: points not core and not reachable from any core point
    is_noise = np.zeros(len(X), dtype=bool)
    for i in range(len(X)):
        if not core_mask[i]:
            is_border = np.any((dist_matrix[i] <= eps) & core_mask)
            if not is_border:
                is_noise[i] = True
                
    return is_noise

# Execute custom outlier detection
noise_mask = detect_density_outliers(X_scaled, eps=0.8, min_samples=5)
print(f'Total observations: {len(X_scaled)}')
print(f'Detected noise points: {np.sum(noise_mask)} ({np.mean(noise_mask)*100:.1f}%)')


Total observations: 200
Detected noise points: 20 (10.0%)


In [3]:
from sklearn.cluster import DBSCAN

# Verify against Scikit-Learn DBSCAN benchmark
dbscan = DBSCAN(eps=0.8, min_samples=5).fit(X_scaled)
sklearn_noise_mask = (dbscan.labels_ == -1)

match = np.array_equal(noise_mask, sklearn_noise_mask)
print(f'Scikit-learn DBSCAN noise points: {np.sum(sklearn_noise_mask)}')
print(f'Custom implementation matches Scikit-learn DBSCAN: {match}')
assert match, 'Mismatch between custom outlier detector and Scikit-Learn DBSCAN!'


Scikit-learn DBSCAN noise points: 20
Custom implementation matches Scikit-learn DBSCAN: True


In [4]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.scatter(
    df.loc[~noise_mask, 'x_coordinate'],
    df.loc[~noise_mask, 'y_coordinate'],
    c='#1f77b4',
    s=60,
    edgecolor='k',
    alpha=0.8,
    label='Dense Cluster Points'
)
plt.scatter(
    df.loc[noise_mask, 'x_coordinate'],
    df.loc[noise_mask, 'y_coordinate'],
    c='#d62728',
    s=120,
    marker='X',
    linewidth=2,
    label=f'Detected Noise Outliers (N={np.sum(noise_mask)})'
)
plt.title('Spatial Density Outlier Detection (Custom vs DBSCAN)', fontsize=13, fontweight='bold')
plt.xlabel('X Coordinate', fontsize=11)
plt.ylabel('Y Coordinate', fontsize=11)
plt.legend(loc='upper right', framealpha=0.9)
plt.grid(True, linestyle='--', alpha=0.5)

output_plot = Path('spatial_density_outliers_plot.png')
plt.savefig(output_plot, dpi=300, bbox_inches='tight')
print(f'Plot saved successfully to: {output_plot}')
try:
    plt.show()
except Exception:
    pass


Plot saved successfully to: spatial_density_outliers_plot.png


C:\Users\conne\AppData\Local\Temp\ipykernel_17840\4003997569.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
